# Deep Learning

# Tutorial 12: CIFAR10 Classification with ResNet

In this tutorial, we will cover:

- How to define convolutional and residual blocks
- How to fine-tune a pretrained model for a new task

Prerequisites:

- Python, Tensor basics, Stochastic Gradient Descent

Our contacts:

- Niklas Beuter (niklas.beuter@th-luebeck.de)
- Fenja Falta (fenja.falta@th-luebeck.de)

Course:

- Slides and notebooks will be available at https://lernraum.th-luebeck.de/course/view.php?id=5383

### Imports

First, we import the necessary modules: `torch` for tensor operations, `torch.nn` for neural network modules, `torch.nn.functional` for activation and loss functions. `torchvision` provides the image datasets, and `tqdm.notebook` creates practical progress bars during training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from tqdm.notebook import tqdm

### Dataset Loading and Preprocessing

This cell loads the CIFAR10 dataset, applies a transformation to convert images to tensors and normalize them. It then creates `DataLoader` objects for both the training and validation sets.

> ## Exercise
> Complete the dataset loading below. As an exercise, we want to use the CIFAR10 dataset from `torchvision.datasets`. Ensure you set the `train` parameter correctly for the training and validation datasets, and decide whether to set `download=True` or `download=False` based on whether you have already downloaded the dataset. Remember to set `train=False` or `train=True` when differentiating between validation and test. Also use the `transform` as defined.

In [ ]:
cifar10_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Standard CIFAR10 normalization
])

# TODO: Load CIFAR10 datasets
cifar10_dataset_train =  # TODO
cifar10_dataset_val = # TODO

print("Loaded CIFAR10 dataset.")

## Dataset Visualization

Let's visualize some images from the training sets. Our aim is to train a neural network that predicts the label based on an image as the input.

In [ ]:
import matplotlib.pyplot as plt

# Visualize CIFAR10 examples
print("Visualizing CIFAR10 examples:")
# Define mean and std for denormalization (matching cifar10_transform)
cifar10_mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
cifar10_std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)

fig_cifar = plt.figure(figsize=(10, 2))
for i in range(5):
    ax = fig_cifar.add_subplot(1, 5, i + 1)
    img_to_show = cifar10_dataset_train[i][0]
    print(f"CIFAR10 image shape: {img_to_show.shape}") # Image shape
    # Denormalize the image: img * std + mean
    img_to_show = img_to_show * cifar10_std + cifar10_mean
    # Clip values to [0, 1] after denormalization, just in case (e.g., due to floating point inaccuracies)
    img_to_show = torch.clamp(img_to_show, 0, 1)
    # Convert from CxHxW to HxWxC for matplotlib
    ax.imshow(img_to_show.permute(1, 2, 0))
    ax.set_title(f"Label: {cifar10_dataset_train[i][1]}")
    ax.axis('off')
fig_cifar.suptitle('Examples of the CIFAR10 training dataset')
plt.tight_layout()
plt.show()

## CNN Model Definition (`CNN`)
> ## Exercise
> Typically, a CNN follows a specific structure. It is built upon blocks that each contain the following layers:
> - 3x3 Convolutional layer (`nn.Conv2d`) that changes the feature dimension
> - Batch normalization (`nn.BatchNorm2d`)
> - ReLU activation (`F.relu`)
> - 3x3 Convolutional layer (`nn.Conv2d`) that retains the feature dimension
> - Batch normalization (`nn.BatchNorm2d`)
> - ReLU activation (`F.relu`)
> - Pooling (`nn.MaxPool2d`)
>
> Those blocks should be defined as its own module `ConvolutionalBlock` that is called in `CNN` to construct the final network.
> Our `CNN` network should consist of two of those blocks with feature dimensions `32` and `64`, followed by a linear layer (`nn.Linear`), that acts as the classifier (don't forget to flatten the extracted features before the linear layer in the `forward` function).

In [ ]:
class ConvolutionalBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvolutionalBlock, self).__init__()
        # TODO
        self.conv1 =
        self.bn1 =
        self.conv2 =
        self.bn2 =
        self.pool =

    def forward(self, x):
        # TODO

        return x

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        # TODO: First ConvolutionalBlock: out_channels=32
        self.block1 =
        # TODO: Second ConvolutionalBlock: out_channels=64
        self.block2 =

        # TODO: Calculate linear layer input size based on CIFAR 32x32 input
        # 32x32 (input) -> pool (16x16 after block1) -> pool (8x8 after block2)
        self.fc =

    def forward(self, x):
        # TODO: First Block

        # TODO: Second Block

        # TODO: Linear Classifier

        return x

## Residual CNN Model Definition (`ResidualBlock` and `ResidualCNN`)
The `ResidualCNN` model builds upon the concept of residual connections to enable deeper networks.
The core component is the `ResidualBlock`, which implements a residual connection that adds the input to the output of a stack of layers.
![](https://upload.wikimedia.org/wikipedia/commons/b/ba/ResBlock.png)
> ## Exercise
> Our `ResidualBlock` should have the same general structure as the `ConvolutionalBlock` above, but we need to incorporate the residual connection in the `forward` function. If the number of channels changes between in- and output, we also need an additional computation (`self.shortcut`) that maps the identity from the number of input channels to the number of output channels. We use a 1x1 convolution followed by a batch normalization for that. You can combine both layers by calling them inside `nn.Sequential`.
>
> Our `ResidualCNN` network should consist of two `ResidualBlock`s, with feature dimensions `32` and `64`, each followed by a pooling layer, and concluded by a linear layer (`nn.Linear`) for classification.

In [ ]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        #TODO: Define the layers of the block (identical to ConvolutionalBlock)
        self.conv1 =
        self.bn1 =
        self.conv2 =
        self.bn2 =

        if in_channels != out_channels:
            self.shortcut = nn.Sequential( #TODO: Define the shortcut (1x1 convolution and batchnorm)

            )
        else:
            self.shortcut = nn.Sequential()


    def forward(self, x):
        #TODO: save the identity
        identity =

        # TODO: forward pass through the layers (conv -> batchnorm -> relu -> conv -> batchnorm)
        out =

        # TODO: Apply shortcut transformation and add to the output
        out = # TODO: Add shortcut
        out = # TODO: Final ReLU for the block output
        return out

class ResidualCNN(nn.Module):
    def __init__(self, num_classes=10, input_channels=3):
        super(ResidualCNN, self).__init__()

        # TODO: First ResidualBlock: out_channels=32
        self.block1 =
        # TODO: Second ResidualBlock: out_channels=62
        self.block2 =
        # TODO: Pooling
        self.pool =
        # TODO: Calculate linear layer input size based on CIFAR 32x32 input
        # 32x32 (input) -> pool (16x16 after block1) -> pool (8x8 after block2)
        self.fc =

    def forward(self, x):
        # TODO: First block

        # TODO: Second block

        # TODO: Linear Classifier

        return x

### Pretrained ResNet Model Definition (`PretrainedResNet`)
> ## Exercise
> Define the `PretrainedResNet` class. This model should leverage a pre-trained ResNet18 model from `torchvision.models`.
> 1. Load the `torchvision.models.resnet18` model and set `pretrained` to `True`.
> 2. Freeze all parameters of the base model to preserve learned features. To do this, iterate over all parameters in the ResNet (`self.model.parameters()`) and set their property `.requires_grad` to `False`.
> 3.  Replace the final fully connected classification layer (self.model.fc) with a new linear layer. You can reuse the number of input features of the original layer (`self.model.fc.in_features`), but need to adapt the output features to CIFAR10.
> 4. In the `forward` function, the forward pass of the ResNet model should be called.

*Remark: The pre-trained ResNets typically expect input images of size 224x224 and normalized with ImageNet means and standard deviations, which will be handled later in the `DataLoader` transformations.*

In [ ]:

class PretrainedResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(PretrainedResNet, self).__init__()
        # TODO: Load a pre-trained ResNet18 model
        self.model =

        # TODO: Freeze all parameters in the network

        # TODO: Replace the final classification layer
        # Get the number of features in the last layer before the classifier
        self.model.fc =

        # The new layer's parameters are trainable by default

    def forward(self, x):
        # TODO

        return

## Model Summaries with `torchsummary`
This cell uses the `torchsummary` library to print a concise summary of each defined neural network model (`CNN`, `ResidualCNN`, and `PretrainedResNet`). The summary provides details about the layers, output shape, and number of parameters for each model, which is useful for understanding their architecture and complexity. The input sizes for the summary are chosen to reflect the typical input dimensions for CIFAR10 (3x32x32 for custom models, 3x224x224 for the pretrained ResNet).

> ## Exercise
> Have a look at the summary and compare the number of (trainable) parameters. How would you expect the models to behave?

In [ ]:
from torchsummary import summary

# Input size for CIFAR10: (channels, height, width)
cifar10_input_size = (3, 32, 32)

# Input size for PretrainedResNet (after resizing): (channels, height, width)
resnet_input_size = (3, 224, 224)

num_classes = 10

print("\n--- CNN Model Summary ---")
cnn_model = CNN(num_classes=num_classes).cuda()
summary(cnn_model, input_size=cifar10_input_size)

print("\n--- ResidualCNN Model Summary ---")
residual_cnn_model = ResidualCNN(num_classes=num_classes, input_channels=cifar10_input_size[0]).cuda()
summary(residual_cnn_model, input_size=cifar10_input_size)

print("\n--- PretrainedResNet Model Summary ---")
pretrained_resnet_model = PretrainedResNet(num_classes=num_classes).cuda()
summary(pretrained_resnet_model, input_size=resnet_input_size)

### Training Function (`train`)
This function encapsulates the training process for a single epoch. It sets the model to training mode (`model.train()`), iterates over the batches in the `train_loader`, performs a forward pass, computes the loss using `F.cross_entropy`, executes a backward pass to calculate gradients, and updates the model's weights using the specified `optimizer`.

## Training Loop

> ## Exercise
> Complete the `train` function below.

In [ ]:
def train(model, train_loader, optimizer, epoch):
    model.train() # TODO: Set model to training mode
    total_loss = 0
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch} Training")):
        # TODO


        total_loss +=  # TODO: Accumulate the loss

    avg_epoch_loss = total_loss / len(train_loader)
    print(f'Train Epoch: {epoch} - Average Loss: {avg_epoch_loss:.6f}')
    return avg_epoch_loss

In [ ]:
def validate(model, val_loader):
    model.eval() #TODO: Set model to evaluation mode.
    val_loss = 0
    correct = 0
    with torch.no_grad(): # Disable gradient calculation for validation
        for data, target in tqdm(val_loader, desc=f"Epoch {epoch} Validation"):
            data, target = data.cuda(), target.cuda() # TODO: Move data and target to GPU
            output = model(data) # TODO: Perform a forward pass
            curr_val_loss = F.cross_entropy(output, target) # TODO: Calculate Cross-Entropy Loss
            pred = output.argmax(dim=1, keepdim=True) # Get the index of the max log-probability as prediction
            correct += pred.eq(target.view_as(pred)).sum().item() # Count correct predictions
            val_loss += curr_val_loss.item()

    val_loss /= len(val_loader) # Calculate average validation loss

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        val_loss, correct, len(val_loader.dataset),
        100. * correct / len(val_loader.dataset))) # Print validation metrics
    return val_loss

Now we can train our network.

In [ ]:
# Define configurations for CNN and ResidualCNN
configurations_cnn_residual = [
    {"dataset": "CIFAR10", "model": "CNN"},
    {"dataset": "CIFAR10", "model": "ResidualCNN"},
]

all_results = [] # Initialize all_results for collecting results from all training runs

print("--- Training CNN and ResidualCNN Models ---")

for config in configurations_cnn_residual:
    dataset_choice = config["dataset"]
    model_choice = config["model"]

    print(f"\n--- Running Configuration: Dataset={dataset_choice}, Model={model_choice} ---")

    # Hardcode CIFAR10 specific parameters for these models
    num_classes = 10
    input_channels = 3

    # Use existing CIFAR10 datasets and their associated transforms
    train_loader = torch.utils.data.DataLoader(cifar10_dataset_train, batch_size=64, shuffle=True)
    val_loader = torch.utils.data.DataLoader(cifar10_dataset_val, batch_size=1000, shuffle=False)

    # Initialize the model based on choice
    if model_choice == "CNN":
        model = CNN(num_classes=num_classes).cuda()
    elif model_choice == "ResidualCNN":
        model = ResidualCNN(num_classes=num_classes).cuda()
    else:
        raise ValueError("Invalid model choice for CNN/ResidualCNN configurations")

    optimizer = torch.optim.Adam(model.parameters())

    current_train_losses = []
    current_val_losses = []

    # TODO: train for x epochs
    for epoch in range(2):
        epoch_train_loss = train(model, train_loader, optimizer, epoch)
        epoch_val_loss = validate(model, val_loader)
        current_train_losses.append(epoch_train_loss)
        current_val_losses.append(epoch_val_loss)

    all_results.append({
        "name": f"{model_choice} - {dataset_choice}",
        "train_losses": current_train_losses,
        "val_losses": current_val_losses
    })

## Training Loop for Pretrained ResNet Model
This block handles the training specifically for the `PretrainedResNet` model on CIFAR10. The model is initialized, an Adam optimizer is set up, and the training and validation functions are executed for a number of epochs.
> ## Exercise
> To be able to use the pretrained ResNet, we need to apply specific transformations required (i.e. resizing to 224x224 and ImageNet normalization) to the dataset.

In [ ]:
configurations_pretrained_resnet = [
    {"dataset": "CIFAR10", "model": "PretrainedResNet"}
]

print("\n--- Training Pretrained ResNet Model ---")

for config in configurations_pretrained_resnet:
    dataset_choice = config["dataset"]
    model_choice = config["model"]

    print(f"\n--- Running Configuration: Dataset={dataset_choice}, Model={model_choice} ---")

    num_classes = 10

    # Transforms for pre-trained models (resize to 224x224, ImageNet normalization)
    pretrained_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(224), # Resize to ResNet's expected input size
        torchvision.transforms.ToTensor(),
        # Normalization using ImageNet's mean and std
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    # TODO: Create new datasets with the pretrained_transform
    cifar10_dataset_train_pt =
    cifar10_dataset_val_pt =

    train_loader = torch.utils.data.DataLoader(cifar10_dataset_train_pt, batch_size=64, shuffle=True)
    val_loader = torch.utils.data.DataLoader(cifar10_dataset_val_pt, batch_size=1000, shuffle=False)

    # Initialize the PretrainedResNet model
    model = PretrainedResNet(num_classes=num_classes).cuda()

    # Print number of parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model_choice}, Number of trainable parameters: {total_params}")

    optimizer = torch.optim.Adam(model.parameters())

    current_train_losses = []
    current_val_losses = []

    # TODO: train for x epochs
    for epoch in range(2):
        epoch_train_loss = train(model, train_loader, optimizer, epoch)
        epoch_val_loss = validate(model, val_loader)
        current_train_losses.append(epoch_train_loss)
        current_val_losses.append(epoch_val_loss)

    all_results.append({
        "name": f"{model_choice} - {dataset_choice}",
        "train_losses": current_train_losses,
        "val_losses": current_val_losses
    })

### Visualize Loss Curves

> ## Exercise
> Train the models for more epochs in the cells above and then visualize the loss curves. (Be careful, as resnet18 takes quite long to train due to its size). Does the loss curve behave as you initially expected?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.figure(figsize=(12, 6))

# Get a colormap for distinct colors for each configuration
colors = plt.get_cmap('rainbow', len(all_results))

for i, result in enumerate(all_results):
    color = colors(i)
    plt.plot(result["train_losses"], label=f'{result["name"]} Training', color=color, linestyle='-')
    plt.plot(result["val_losses"], label=f'{result["name"]} Validation', color=color, linestyle='--')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Losses for All Configurations')
plt.legend()
plt.grid(True)
plt.show()